# 🤖 Building AI Agents with Google ADK

## A Step-by-Step Guide to Multi-Agent Systems

---

### 📋 **What You'll Learn**

This notebook provides a **technical deep-dive** into building intelligent agents using Google's Agent Development Kit (ADK). We'll walk through the complete process of creating our 4-agent system for accessible services navigation.

**Topics Covered:**
1. ADK Architecture & Core Concepts
2. Creating Custom Tools
3. Building Specialized Agents
4. Pydantic Models for Type Safety
5. Agent Integration & Handoffs
6. Testing Individual Agents

**Prerequisites:**
- Basic Python knowledge
- Understanding of LLMs and prompting
- Familiarity with async/await (helpful but not required)

---

### 🎯 **Our Goal**

By the end of this notebook, you'll understand how to:
- ✅ Design agents with clear responsibilities
- ✅ Create custom tools for specialized tasks
- ✅ Structure prompts for consistent outputs
- ✅ Validate data with Pydantic schemas
- ✅ Test agents independently

Let's build! 🚀

## 🏗️ ADK Architecture Fundamentals

Google's Agent Development Kit (ADK) provides three core abstractions:

### 1. **Agents** 🤖
Autonomous entities that can:
- Process natural language
- Use tools
- Maintain conversation context
- Generate structured outputs

### 2. **Tools** 🛠️
Functions that agents can call to:
- Access external data
- Perform computations
- Interact with APIs
- Retrieve information

### 3. **Memory** 💾
Storage for:
- Session-level data (current conversation)
- Long-term data (user profiles, history)
- Conversation context

---

### Our 4-Agent Architecture

```
Query → [Intake] → UserProfile
          ↓
       [Search] → CandidateFacilities  
          ↓
      [Reasoning] → ScoredFacilities
          ↓
   [Recommendation] → ServicePlan
```

Each agent specializes in ONE task!

## 📦 Setup: Environment and Imports

Let's set up our development environment and import all necessary modules.

In [ ]:
# Navigate to implementation directory
import sys
import os
from pathlib import Path

if 'implementation' not in os.getcwd():
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

print(f"✅ Working directory: {os.getcwd()}")

In [ ]:
# Core imports
import asyncio
from typing import Dict, List, Any, Optional
from pydantic import BaseModel, Field
import pandas as pd

# Google ADK imports
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.genai import types

# Our custom modules
from src.models.schemas import (
    UserProfile, 
    Facility, 
    CandidateFacilities,
    ScoredFacility,
    ServicePlan,
    DisabilityType,
    ServiceCategory
)

print("✅ All imports successful!")

## 📊 Pydantic Schemas: Type Safety for Agent Communication

**Why Pydantic?**
- ✅ **Type validation**: Catches errors before they reach production
- ✅ **Auto-documentation**: Self-documenting data structures
- ✅ **JSON serialization**: Easy to save/load from databases
- ✅ **IDE support**: Autocomplete and type hints

Let's examine the key schemas used in our system.

In [ ]:
# Example 1: UserProfile Schema
# This is what the Intake Agent produces

example_profile = UserProfile(
    disability_type=DisabilityType.MOBILITY,
    mobility_needs="wheelchair user",
    communication_needs=None,
    preferred_subcounty="Embakasi East",
    backup_subcounty="Embakasi West",
    service_category=ServiceCategory.CLINIC,
    additional_requirements="affordable, regular check-ups",
    language_preference="English"
)

print("🎯 UserProfile Example:")
print("="*60)
print(example_profile.model_dump_json(indent=2))
print("\n💡 Key Benefit: All downstream agents receive validated, structured data!")

In [ ]:
# Example 2: Facility Schema
# This represents a single facility in our dataset

example_facility = Facility(
    facility_id="CLN-003",
    facility_name="Westlands Primary Care Clinic",
    category=ServiceCategory.CLINIC,
    subcounty="Westlands",
    ward="Karura",
    neighbourhood_landmark="near Sarit Centre",
    latitude=-1.2647,
    longitude=36.8045,
    managing_agency="Private (low-cost)",
    services_offered="general consultation|chronic disease management|lab services",
    has_ramp=True,
    has_elevator_or_step_free_entry=True,
    has_accessible_toilet=True,
    has_sign_language_support=True,
    supports_text_based_contact=True,
    visual_signage_quality="high",
    crowding_level="low",
    approx_cost_level="low",
    mobility_score=3,
    hearing_score=3,
    visual_score=3,
    notes="Excellent accessibility. Modern facility.",
    data_source="NGO survey",
    last_verified_date="2024-11-10"
)

print("🏥 Facility Schema Example:")
print("="*60)
# Display key fields
print(f"Name: {example_facility.facility_name}")
print(f"Location: {example_facility.subcounty}, {example_facility.ward}")
print(f"Accessibility: Ramp={example_facility.has_ramp}, Elevator={example_facility.has_elevator_or_step_free_entry}")
print(f"Scores: Mobility={example_facility.mobility_score}, Hearing={example_facility.hearing_score}, Visual={example_facility.visual_score}")
print(f"\n💡 This schema has {len(example_facility.model_fields)} fields with full validation!")

## 🛠️ Custom Tools: DatasetSearchTool

Tools are functions that agents can call to perform specific tasks. Let's create a custom tool that searches our facilities dataset.

**Key Concepts:**
- Tools extend agent capabilities beyond language generation
- Tools must have clear inputs/outputs
- Tools should be single-purpose and reusable
- Tools can access external data (databases, APIs, files)

In [ ]:
# Import the DatasetSearchTool
from src.tools.dataset_search import DatasetSearchTool

# Initialize the tool
search_tool = DatasetSearchTool()

print("✅ DatasetSearchTool initialized")
print(f"📊 Total facilities loaded: {len(search_tool.all_facilities_df)}")
print(f"   🏥 Clinics: {len(search_tool.clinics_df)}")
print(f"   🏢 Social Services: {len(search_tool.social_services_df)}")

In [ ]:
# Test the DatasetSearchTool with a sample user profile
test_profile = UserProfile(
    disability_type=DisabilityType.MOBILITY,
    mobility_needs="wheelchair user",
    preferred_subcounty="Westlands",
    service_category=ServiceCategory.CLINIC,
    additional_requirements="accessible toilet"
)

# Perform a search
results = search_tool.search_facilities(test_profile, max_results=5)

print("🔍 Search Results:")
print("="*60)
print(f"✅ Found {results.count} facilities")
print(f"\n📊 Search Metadata:")
for key, value in results.search_metadata.items():
    print(f"   {key}: {value}")

print(f"\n🏥 Top Facilities:")
for i, facility in enumerate(results.facilities[:3], 1):
    print(f"\n{i}. {facility.facility_name}")
    print(f"   📍 {facility.subcounty}, {facility.ward}")
    print(f"   ♿ Mobility Score: {facility.mobility_score}/3")
    print(f"   💰 Cost: {facility.approx_cost_level}")

## 🤖 Agent 1: Intake Agent

The **Intake Agent** is responsible for understanding user queries and extracting structured information.

**Key Responsibilities:**
1. Greet users warmly
2. Extract disability type, location, and service needs
3. Ask minimal clarifying questions
4. Output structured UserProfile

**Prompt Engineering Tips:**
- Clear role definition
- Specific output format expectations
- Examples of good behavior
- Guardrails for edge cases

In [ ]:
# Import the Intake Agent creator
from src.agents.intake_agent import create_intake_agent

# Create the agent
intake_agent = create_intake_agent(
    model_name="gemini-2.0-flash-exp",
    temperature=0.7
)

print("✅ Intake Agent created!")
print(f"🤖 Name: {intake_agent.name}")
print(f"📝 Description: {intake_agent.description}")
print(f"\n🎯 Instruction Preview:")
print("="*60)
print(intake_agent.instruction[:300] + "...")
print("="*60)

In [ ]:
# Test the Intake Agent independently
test_query = "I use a wheelchair and need a clinic in Embakasi East that's affordable."

print("📝 User Query:", test_query)
print("\n⏳ Processing...")

# Simulate agent interaction (in production, this would be async)
response = await intake_agent.execute(input_message=test_query)

print("\n✅ Intake Agent Response:")
print("="*60)
print(response.content)
print("="*60)
print("\n💡 Notice how the agent extracts structured data from natural language!")

## 🤖 Agent 2: Search Agent

The **Search Agent** uses the UserProfile to find matching facilities from the dataset.

**Key Responsibilities:**
1. Receive UserProfile from Intake Agent
2. Use DatasetSearchTool to filter facilities
3. Apply location and accessibility filters
4. Return CandidateFacilities

**Why Separate Search Agent?**
- Separation of concerns (intake vs. retrieval)
- Can swap search implementations (database, API, etc.)
- Easier to test and debug
- Can cache search results

In [ ]:
# Import the Search Agent creator
from src.agents.search_agent import create_search_agent

# Create the agent with the search tool
search_agent = create_search_agent(
    search_tool=search_tool,
    model_name="gemini-2.0-flash-exp"
)

print("✅ Search Agent created!")
print(f"🤖 Name: {search_agent.name}")
print(f"🛠️  Tools available: {len(search_agent.tools) if hasattr(search_agent, 'tools') else 'Integrated'}")
print(f"\n🎯 Key Function: Filters {len(search_tool.all_facilities_df)} facilities → Top matches")

## 🤖 Agent 3: Reasoning Agent

The **Reasoning Agent** scores and ranks facilities based on multiple criteria.

**Key Responsibilities:**
1. Receive CandidateFacilities from Search Agent
2. Score each facility on:
   - Accessibility match (disability-specific)
   - Location relevance
   - Cost appropriateness
   - Service quality
3. Provide justification for each score
4. Rank facilities by overall score

**Scoring Algorithm:**
- Base accessibility score (0-3 from dataset)
- Location bonus (preferred vs. backup)
- Cost alignment (free > low > moderate)
- Feature completeness (ramp + toilet + signage)

In [ ]:
# Import the Reasoning Agent creator
from src.agents.reasoning_agent import create_reasoning_agent

# Create the agent
reasoning_agent = create_reasoning_agent(
    model_name="gemini-2.0-flash-exp",
    temperature=0.3  # Lower temperature for more consistent scoring
)

print("✅ Reasoning Agent created!")
print(f"🤖 Name: {reasoning_agent.name}")
print(f"🧠 Temperature: 0.3 (for consistent scoring)")
print(f"\n🎯 Key Function: Scores facilities on 0-10 scale with justification")

## 🤖 Agent 4: Recommendation Agent

The **Recommendation Agent** creates the final personalized service plan.

**Key Responsibilities:**
1. Receive ScoredFacilities from Reasoning Agent
2. Generate human-readable recommendations
3. Include practical guidance:
   - How to get there (landmarks, public transport)
   - What to bring (documents, NHIF card)
   - Best times to visit (avoid crowds)
   - Contact information
4. Provide fallback options

**Output Style:**
- Warm and encouraging tone
- Action-oriented language
- Clear step-by-step guidance
- Accessibility-focused tips

In [ ]:
# Import the Recommendation Agent creator
from src.agents.recommendation_agent import create_recommendation_agent

# Create the agent
recommendation_agent = create_recommendation_agent(
    model_name="gemini-2.0-flash-exp",
    temperature=0.8  # Higher temperature for more natural language
)

print("✅ Recommendation Agent created!")
print(f"🤖 Name: {recommendation_agent.name}")
print(f"📝 Temperature: 0.8 (for natural, varied language)")
print(f"\n🎯 Key Function: Generates actionable service plans with guidance")

## 🧪 Testing the Complete Pipeline

Now let's test all 4 agents working together through the orchestrator.

**Test Scenario:**
- User: Wheelchair user in Langata
- Need: Affordable clinic
- Requirements: Ramp, accessible toilet

In [ ]:
# Import the orchestrator
from src.orchestrator import AgentOrchestrator

# Initialize orchestrator (creates all agents + memory)
orchestrator = AgentOrchestrator()

print("✅ Orchestrator initialized!")
print("🤖 All 4 agents created and connected")
print("💾 Memory manager ready")
print("\n" + "="*60)

In [ ]:
# Run a complete test
test_query = "I'm a wheelchair user in Langata. Need affordable clinic with ramps."
test_user_id = "test_user_dev_notebook"

print(f"📝 Query: {test_query}")
print(f"👤 User ID: {test_user_id}")
print("\n⏳ Running 4-agent pipeline...")
print("="*60)

result = await orchestrator.process_query(
    user_id=test_user_id,
    query=test_query
)

print("\n✅ Pipeline complete!")
print(f"⏱️  Total time: {result.get('total_time_seconds', 0):.2f}s")
print(f"📊 Facilities found: {len(result.get('candidate_facilities', []))}")
print(f"🏆 Facilities scored: {len(result.get('scored_facilities', []))}")
print("\n📋 Final Recommendation:")
print("="*60)
print(result.get('recommendation', 'No recommendation generated'))
print("="*60)

## 🎯 Key Takeaways

### What We Learned

**1. Agent Design Principles** 🤖
- **Single Responsibility**: Each agent does ONE thing well
- **Clear Contracts**: Pydantic schemas define inputs/outputs
- **Composability**: Agents can be swapped or extended independently

**2. Tool Creation** 🛠️
- Tools extend agent capabilities beyond language
- Tools should be stateless and reusable
- Tools provide deterministic operations (search, calculation, API calls)

**3. Prompt Engineering** 📝
- Clear role definition + specific output format
- Examples of desired behavior
- Guardrails for edge cases
- Temperature tuning (low for consistency, high for creativity)

**4. Testing Strategy** 🧪
- Test agents individually before integration
- Use synthetic data for edge cases
- Monitor timing and token usage
- Validate schema compliance

---

### Next Steps

- **Notebook 01**: Explore the dataset creation process
- **Notebook 03**: See the full system in action
- **Notebook 04**: Learn about evaluation and metrics

**Try modifying:**
- Agent prompts (change tone, add features)
- Scoring algorithms (different weights)
- Tool implementations (add caching, web search)
- Schema validation (stricter rules)